## Bước 1: Profiling dữ liệu trước khi clean.
Mục đích: hiểu thực tế dataset đang có vấn đề gì, trước khi viết logic clean.

In [ ]:
import pandas as pd

CSV_PATH = "../data/raw/ecommerce_sales_analytics_5000.csv"  
def main():
    df = pd.read_csv(CSV_PATH)

    print("=" * 60)
    print("1. TỔNG QUAN")
    print("=" * 60)
    print(f"Số dòng: {len(df)}")
    print(f"Số cột: {len(df.columns)}")
    print(f"Các cột: {list(df.columns)}\n")

    print("=" * 60)
    print("2. KIỂU DỮ LIỆU HIỆN TẠI")
    print("=" * 60)
    print(df.dtypes, "\n")

    print("=" * 60)
    print("3. MISSING VALUES")
    print("=" * 60)
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    print(pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}), "\n")

    print("=" * 60)
    print("4. DUPLICATE")
    print("=" * 60)
    print(f"Số dòng trùng hoàn toàn: {df.duplicated().sum()}")
    if "order_id" in df.columns:
        print(f"Số order_id trùng: {df['order_id'].duplicated().sum()}\n")

    print("=" * 60)
    print("5. GIÁ TRỊ ÂM / BẤT THƯỜNG (các cột số)")
    print("=" * 60)
    numeric_cols = df.select_dtypes(include="number").columns
    for col in numeric_cols:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"  {col}: {neg_count} giá trị âm")
    print()

    print("=" * 60)
    print("6. THỐNG KÊ MÔ TẢ (numeric)")
    print("=" * 60)
    print(df.describe(), "\n")

    print("=" * 60)
    print("7. GIÁ TRỊ DUY NHẤT (categorical)")
    print("=" * 60)
    cat_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in cat_cols:
        if col != "order_date":
            print(f"  {col}: {df[col].unique()}")
    print()

    print("=" * 60)
    print("8. KIỂM TRA revenue = quantity * unit_price * (1 - discount)")
    print("=" * 60)
    if all(c in df.columns for c in ["quantity", "unit_price", "discount", "revenue"]):
        expected = df["quantity"] * df["unit_price"] * (1 - df["discount"])
        diff = (df["revenue"] - expected).abs()
        mismatches = (diff > 0.05).sum()  # sai lệch > 0.05 do làm tròn
        print(f"Số dòng revenue KHÔNG khớp công thức: {mismatches} / {len(df)}")
        if mismatches > 0:
            print(df.loc[diff > 0.05, ["order_id", "quantity", "unit_price", "discount", "revenue"]].head())

if __name__ == "__main__":
    main()

1. TỔNG QUAN
Số dòng: 5000
Số cột: 12
Các cột: ['order_id', 'order_date', 'customer_id', 'product_category', 'region', 'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days', 'customer_rating', 'revenue']

2. KIỂU DỮ LIỆU HIỆN TẠI
order_id              int64
order_date              str
customer_id           int64
product_category        str
region                  str
quantity              int64
unit_price          float64
discount            float64
payment_method          str
delivery_days         int64
customer_rating     float64
revenue             float64
dtype: object 

3. MISSING VALUES
                  missing_count  missing_pct
order_id                      0          0.0
order_date                    0          0.0
customer_id                   0          0.0
product_category              0          0.0
region                        0          0.0
quantity                      0          0.0
unit_price                    0          0.0
discount              

/tmp/ipykernel_29504/2471728893.py:52: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns
